In [43]:
import os
print(os.getcwd())
print(os.listdir())

/Users/hannah/projects/data_analytics_projects/supermarket_sales_sql_analysis
['.DS_Store', 'images', 'supermarket_sales_sql_analysis.ipynb', 'README.md', 'data']


In [44]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [45]:
# Load dataset into pandas
import pandas as pd

df = pd.read_csv('data/supermarket_sales.csv')
df.head()

,Invoice ID,Branch,Customer type,Gender,Product line,Unit price,Quantity,Date,Time,Payment,Rating,City,Total
0,0202-7941,A,Normal,Male,Food and beverages,68.34,2,01/20/2025,16:42,Ewallet,5.9,Yangon,143.51
1,0535-8002,B,Member,Male,Home and lifestyle,30.83,1,02/03/2025,20:40,Ewallet,9.2,Mandalay,32.37
2,0960-2331,B,Normal,Male,Fashion accessories,71.32,9,01/25/2025,18:01,Cash,9.3,Mandalay,673.97
3,0370-5261,C,Member,Female,Sports and travel,94.62,9,02/17/2025,16:51,Cash,6.1,Naypyitaw,894.16
4,0206-8648,A,Member,Male,Sports and travel,14.21,7,02/25/2025,16:08,Credit card,4.5,Yangon,104.44


In [46]:
# Initialize DuckDB and register DataFrame as a table
import duckdb

con = duckdb.connect()
con.register("sales", df)

In [47]:
# Optional: check schema and row count
# con.execute("DESCRIBE sales;").df()
# con.execute("SELECT COUNT(*) AS n_rows FROM sales;").df()

# Basic structure check
con.execute("SELECT * FROM sales LIMIT 10;").df()

,Invoice ID,Branch,Customer type,Gender,Product line,Unit price,Quantity,Date,Time,Payment,Rating,City,Total
0,0202-7941,A,Normal,Male,Food and beverages,68.34,2,01/20/2025,16:42,Ewallet,5.9,Yangon,143.51
1,0535-8002,B,Member,Male,Home and lifestyle,30.83,1,02/03/2025,20:40,Ewallet,9.2,Mandalay,32.37
2,0960-2331,B,Normal,Male,Fashion accessories,71.32,9,01/25/2025,18:01,Cash,9.3,Mandalay,673.97
3,0370-5261,C,Member,Female,Sports and travel,94.62,9,02/17/2025,16:51,Cash,6.1,Naypyitaw,894.16
4,0206-8648,A,Member,Male,Sports and travel,14.21,7,02/25/2025,16:08,Credit card,4.5,Yangon,104.44
5,0171-6115,A,Member,Female,Health and beauty,38.77,7,03/05/2025,19:12,Cash,8.3,Yangon,284.96
6,0800-5637,C,Member,Female,Home and lifestyle,49.44,3,01/11/2025,10:23,Ewallet,5.9,Naypyitaw,155.74
7,0120-5854,A,Normal,Male,Fashion accessories,90.93,10,03/30/2025,12:23,Ewallet,8.7,Yangon,954.77
8,0714-4177,B,Member,Female,Fashion accessories,96.60,2,01/03/2025,12:34,Ewallet,6.7,Mandalay,202.86
9,0221-8730,C,Normal,Female,Electronic accessories,49.97,7,01/19/2025,17:33,Ewallet,6.9,Naypyitaw,367.28


In [48]:
# Quick field-level sanity check
con.execute("""SELECT Date, Branch, "Invoice ID" FROM sales LIMIT 5;""").df()

,Date,Branch,Invoice ID
0,01/20/2025,A,0202-7941
1,02/03/2025,B,0535-8002
2,01/25/2025,B,0960-2331
3,02/17/2025,C,0370-5261
4,02/25/2025,A,0206-8648


In [49]:
# Step 1: Compare weekday vs weekend traffic by branch
# Metric: average daily transaction count

con.execute("""
WITH base AS (
    -- Parse date once
    SELECT
        "Invoice ID",
        Branch,
        STRPTIME(Date, '%m/%d/%Y') AS dt
    FROM sales
),

feat AS (
    -- Derive weekday/weekend classification
    SELECT
        "Invoice ID",
        Branch,
        dt,
        CASE
            WHEN STRFTIME(dt, '%w') IN ('0','6') THEN 'Weekend'
            ELSE 'Weekday'
        END AS day_type
    FROM base
),

agg_daily AS (
    -- Aggregate to daily transaction level
    SELECT
        dt,
        Branch,
        day_type,
        COUNT(DISTINCT "Invoice ID") AS daily_transactions
    FROM feat
    GROUP BY dt, Branch, day_type
)

-- Compare average daily traffic by branch and day type
SELECT
    Branch,
    day_type,
    AVG(daily_transactions) AS avg_daily_transactions,
    COUNT(*) AS n_days
FROM agg_daily
GROUP BY Branch, day_type
ORDER BY Branch, day_type;
""").df()

,Branch,day_type,avg_daily_transactions,n_days
0,A,Weekday,3.888889,63
1,A,Weekend,3.615385,26
2,B,Weekday,3.857143,63
3,B,Weekend,4.192308,26
4,C,Weekday,3.507937,63
5,C,Weekend,3.520000,25


In [50]:
# Step 2: Identify peak time slots by traffic (Top N)
# Metric: transaction count (traffic)

con.execute("""
WITH base AS (
    -- Parse date and time once (avoid repeated conversion)
    SELECT
        "Invoice ID",
        STRPTIME(Date, '%m/%d/%Y') AS dt,
        STRPTIME(Time, '%H:%M')    AS tm,
        Total
    FROM sales
),

feat AS (
    -- Derive weekday/weekend classification and hour
    SELECT
        "Invoice ID",
        STRFTIME(dt, '%A') AS day_name,
        CASE
            WHEN STRFTIME(dt, '%w') IN ('0','6') THEN 'Weekend'
            ELSE 'Weekday'
        END AS day_type,
        EXTRACT(HOUR FROM tm) AS hour,
        Total
    FROM base
),

agg_hourly AS (
    -- Aggregate to hourly level (day_name × hour)
    SELECT
        day_name,
        day_type,
        hour,
        COUNT(DISTINCT "Invoice ID") AS transaction_count,
        SUM(Total) AS revenue,
        AVG(Total) AS atv
    FROM feat
    GROUP BY day_name, day_type, hour
)

-- Rank slots by transaction_count and return Top N
SELECT
    day_name, day_type, hour, transaction_count, revenue, atv
FROM agg_hourly
ORDER BY transaction_count DESC
LIMIT 10;
""").df()

,day_name,day_type,hour,transaction_count,revenue,atv
0,Saturday,Weekend,15,27,11032.71,408.618889
1,Thursday,Weekday,16,24,4468.25,186.177083
2,Tuesday,Weekday,19,21,6393.47,304.450952
3,Thursday,Weekday,11,20,5996.59,299.829500
4,Wednesday,Weekday,16,20,7869.98,393.499000
5,Monday,Weekday,18,19,6630.08,348.951579
6,Monday,Weekday,15,19,7176.58,377.714737
7,Friday,Weekday,20,18,5365.96,298.108889
8,Sunday,Weekend,17,18,4368.52,242.695556
9,Saturday,Weekend,11,17,5144.36,302.609412


In [51]:
# Step 3: Validate peak driver (transaction_count vs ATV)
# Metric: peak vs baseline ratio comparison

con.execute("""
WITH base AS (
    -- Parse date and time once (avoid repeated conversion)
    SELECT
        "Invoice ID",
        STRPTIME(Date, '%m/%d/%Y') AS dt,
        STRPTIME(Time, '%H:%M')    AS tm,
        Total
    FROM sales
),

feat AS (
    -- Derive day_name and hour features
    SELECT
        "Invoice ID",
        STRFTIME(dt, '%A') AS day_name,
        EXTRACT(HOUR FROM tm) AS hour,
        Total
    FROM base
),

agg_hourly AS (
    -- Aggregate to (day_name × hour) level
    SELECT
        day_name,
        hour,
        COUNT(DISTINCT "Invoice ID") AS transaction_count,
        AVG(Total) AS atv
    FROM feat
    GROUP BY day_name, hour
),

baseline AS (
    -- Compute overall average across all time slots
    SELECT
        AVG(transaction_count) AS avg_transaction_count,
        AVG(atv)     AS avg_atv
    FROM agg_hourly
),

peak AS (
    -- Peak slot identified in Step 2 (Saturday 15:00)
    SELECT
        transaction_count,
        atv
    FROM agg_hourly
    WHERE day_name = 'Saturday' AND hour = 15
)

-- Compare peak vs baseline
SELECT
    peak.transaction_count,
    baseline.avg_transaction_count,
    ROUND(peak.transaction_count / baseline.avg_transaction_count, 2) AS transaction_count_ratio,
    peak.atv,
    baseline.avg_atv,
    ROUND(peak.atv / baseline.avg_atv, 2) AS atv_ratio
FROM peak
CROSS JOIN baseline;
""").df()

,transaction_count,avg_transaction_count,transaction_count_ratio,atv,avg_atv,atv_ratio
0,27,12.987013,2.08,408.618889,312.842198,1.31


In [52]:
# Step 4: Compare top transaction_count slot vs top ATV slot (day_name × hour)
# Metric: top-1 by transaction_count vs top-1 by ATV (do they match?)

con.execute("""
WITH base AS (
    -- Parse date/time once (avoid repeated conversion)
    SELECT
        STRPTIME(Date, '%m/%d/%Y') AS dt,
        STRPTIME(Time, '%H:%M')    AS tm,
        "Invoice ID" AS invoice_id,
        Total
    FROM sales
),

feat AS (
    -- Derive day_name and hour features
    SELECT
        STRFTIME(dt, '%A') AS day_name,
        EXTRACT(HOUR FROM tm) AS hour,
        invoice_id,
        Total
    FROM base
),

agg_hourly AS (
    -- Aggregate to (day_name × hour) level
    SELECT
        day_name,
        hour,
        COUNT(DISTINCT invoice_id) AS transaction_count,
        SUM(Total) AS revenue,
        AVG(Total) AS atv
    FROM feat
    GROUP BY day_name, hour
),

ranked AS (
    -- Rank time slots by transaction_count and by ATV (top 1 each)
    SELECT
        day_name,
        hour,
        transaction_count,
        revenue,
        atv,
        ROW_NUMBER() OVER (ORDER BY transaction_count DESC) AS r_transaction_count,
        ROW_NUMBER() OVER (ORDER BY atv DESC)     AS r_atv
    FROM agg_hourly
)

-- Show the top transaction_count slot and top ATV slot side by side
SELECT
    CASE
        WHEN r_transaction_count = 1 THEN 'Top Traffic'
        WHEN r_atv = 1 THEN 'Top ATV'
    END AS top_type,
    day_name,
    hour,
    transaction_count,
    revenue,
    atv
FROM ranked
WHERE r_transaction_count = 1 OR r_atv = 1
ORDER BY top_type;
""").df()

,top_type,day_name,hour,transaction_count,revenue,atv
0,Top ATV,Friday,15,10,4851.69,485.169000
1,Top Traffic,Saturday,15,27,11032.71,408.618889
